In [23]:
import numpy as np
import pandas as pd
import os

!pip install mlflow dagshub xgboost -q

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("DAGSHUB_USERNAME")

import mlflow
mlflow.set_tracking_uri("https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow")
mlflow.set_experiment("XGBoost_Training")

print("MLflow connected!")

MLflow connected!


In [24]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

train_transaction = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_identity = pd.read_csv(f"{DATA_DIR}/train_identity.csv")

train = train_transaction.merge(train_identity, on="TransactionID", how="left")

del train_transaction, train_identity
import gc
gc.collect()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f} ({train['isFraud'].sum()} / {len(train)})")
print(f"Columns: {train.shape[1]}")

Train shape: (590540, 434)
Fraud rate: 0.0350 (20663 / 590540)
Columns: 434


# EDA

პირველ რიგში, შევხედოთ მონაცემებს და გავარკვიოთ რამდენია კატეგორიული ცვლადი, რამდენია ცარიელი, რომელ ფიჩერებს აქვთ ყველაზე მეტი ცარიელი მნიშვნელობა

In [25]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = (
    pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
    .query("n_missing > 0")
    .sort_values("pct_missing", ascending=False)
)
print(f"\nColumns with any missing values: {len(missing_df)} out of {train.shape[1]}")
print(f"\nColumns with >50% missing:")
print(missing_df[missing_df["pct_missing"] > 50].shape[0])
print(f"\nTop 20 most missing:")
print(missing_df.head(20))

Numerical columns: 403
Categorical columns: 31

Columns with any missing values: 414 out of 434

Columns with >50% missing:
214

Top 20 most missing:
       n_missing  pct_missing
id_24     585793        99.20
id_26     585377        99.13
id_25     585408        99.13
id_21     585381        99.13
id_07     585385        99.13
id_08     585385        99.13
id_23     585371        99.12
id_22     585371        99.12
id_27     585371        99.12
dist2     552913        93.63
D7        551623        93.41
id_18     545427        92.36
D13       528588        89.51
D14       528353        89.47
D12       525823        89.04
id_04     524216        88.77
id_03     524216        88.77
D6        517353        87.61
id_33     517251        87.59
id_09     515614        87.31


# Data Separation
მონაცემების 80/20 გაყოფა train და validation სეტებად. stratify პარამეტრით ვინარჩუნებთ fraud-ის 3.5% თანაფარდობას ორივე ნაწილში.

In [26]:
from sklearn.model_selection import train_test_split

y = train["isFraud"]
X = train.drop(columns=["isFraud", "TransactionID"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"Fraud rate train: {y_train.mean():.4f}")
print(f"Fraud rate val:   {y_val.mean():.4f}")

del train
gc.collect()

X_train: (472432, 432)
X_val:   (118108, 432)
Fraud rate train: 0.0350
Fraud rate val:   0.0350


3

# Cleaning
ვშლით 90%-ზე მეტი null-ის მქონე სვეტებს. დანარჩენ null-ებისთვის რიცხვითებს ვავსებთ -999-ით, ხოლო კატეგორიილებს ტექსტით "missing"

In [27]:
from sklearn.preprocessing import LabelEncoder

missing_pct = X_train.isnull().sum() / len(X_train)
high_null_cols = missing_pct[missing_pct > 0.9].index.tolist()
print(f"Dropping {len(high_null_cols)} columns with >90% nulls")

X_train = X_train.drop(columns=high_null_cols)
X_val = X_val.drop(columns=high_null_cols)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Remaining: {len(num_cols)} numeric, {len(cat_cols)} categorical")

X_train[num_cols] = X_train[num_cols].fillna(-999)
X_val[num_cols] = X_val[num_cols].fillna(-999)

X_train[cat_cols] = X_train[cat_cols].fillna("missing")
X_val[cat_cols] = X_val[cat_cols].fillna("missing")

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_val[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    label_encoders[col] = le

print(f"Final shape: {X_train.shape}")
print(f"Remaining NaN: {X_train.isnull().sum().sum()}")
print(f"Remaining object cols: {(X_train.dtypes == 'object').sum()}")

Dropping 12 columns with >90% nulls
Remaining: 391 numeric, 29 categorical
Final shape: (472432, 420)
Remaining NaN: 0
Remaining object cols: 0


# Feature Engineering
უკვე არსებული ცვლადებიდან გამოგვყავს 7 ახალი ცვლადი, მაგალითად ტრანზაქციის საათი, მისი ათობითი ნაწილი, ბარათზე ტრანზაქციების საშუალო და ა.შ.

In [28]:
for df in [X_train, X_val]:
    df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
    df["Transaction_dow"] = (df["TransactionDT"] / 86400) % 7
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
    df["TransactionAmt_decimal"] = (df["TransactionAmt"] - df["TransactionAmt"].astype(int)).round(2)
    df["Card1_count"] = df["card1"].map(df["card1"].value_counts())
    df["Card1_TransactionAmt_mean"] = df["card1"].map(df.groupby("card1")["TransactionAmt"].mean())
    df["Amt_div_card1mean"] = df["TransactionAmt"] / (df["Card1_TransactionAmt_mean"] + 1)

new_features = [
    "Transaction_hour", "Transaction_dow", "TransactionAmt_log",
    "TransactionAmt_decimal", "Card1_count", "Card1_TransactionAmt_mean",
    "Amt_div_card1mean",
]
print(f"Added {len(new_features)} features")
print(f"X_train shape: {X_train.shape}")

Added 7 features
X_train shape: (472432, 427)


/tmp/ipykernel_57/2669828813.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
/tmp/ipykernel_57/2669828813.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Transaction_dow"] = (df["TransactionDT"] / 86400) % 7
/tmp/ipykernel_57/2669828813.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To 

# Feature Selection
შევიყვანოთ მონაცემები მცირე XGBOOST მოდელში, რათა გავიგოთ, რომელი ფიჩერებია მნიშვნელოვანი, და დავტოვოთ მხოლოდ ისენი. 

In [29]:
from xgboost import XGBClassifier

quick_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="auc",
    use_label_encoder=False,
    n_jobs=-1,
)
quick_model.fit(X_train, y_train)

importances = pd.Series(
    quick_model.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)

top_n = 200
selected_features = importances.head(top_n).index.tolist()

print(f"Selected top {top_n} features from {X_train.shape[1]}")
print(f"\nTop 15:")
print(importances.head(15))
print(f"\nBottom 5 (unused):")
print(importances[importances == 0].shape[0], "features with zero importance")

X_train_sel = X_train[selected_features]
X_val_sel = X_val[selected_features]

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:27:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Selected top 200 features from 427

Top 15:
V258     0.246623
V149     0.043588
V201     0.039840
V70      0.034856
V91      0.034263
V295     0.030577
V147     0.020643
C8       0.017424
V294     0.016270
id_17    0.015791
C12      0.014891
C14      0.014321
V29      0.013904
V308     0.013067
V283     0.010797
dtype: float32

Bottom 5 (unused):
140 features with zero importance


# Training
ვატრენინგებთ XGBoost-ს სხვადასხვა კონფიგურაციით.

In [30]:
from sklearn.metrics import roc_auc_score

def train_and_log_xgb(run_name, params, X_tr, X_va, y_tr, y_va):
    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",
        device="cuda",
        n_jobs=-1,
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )

    train_pred = model.predict_proba(X_tr)[:, 1]
    val_pred = model.predict_proba(X_va)[:, 1]

    train_auc = roc_auc_score(y_tr, train_pred)
    val_auc = roc_auc_score(y_va, val_pred)
    gap = train_auc - val_auc

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("n_features", X_tr.shape[1])
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("overfit_gap", gap)
        mlflow.xgboost.log_model(model, name="model")

    print(f"{run_name:40s}  train_auc={train_auc:.4f}  val_auc={val_auc:.4f}  gap={gap:+.4f}")
    return model, val_auc, val_pred

## Baseline
სტანდარტული XGBOOST

In [32]:
params_baseline = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
}

model_baseline, auc_baseline, _ = train_and_log_xgb(
    "XGBoost_baseline",
    params_baseline,
    X_train_sel, X_val_sel, y_train, y_val,
)

🏃 View run XGBoost_baseline at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/16806ffea1974e008ed17ddfebcf78cf
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
XGBoost_baseline                          train_auc=0.9643  val_auc=0.9411  gap=+0.0232


## Overfit ტესტი
მეტი ხე და მეტი სიღრმე, რათა ვნახოთ გვექნება თუ არა ოვერფიტი

In [31]:
params_overfit = {
    "n_estimators": 1000,
    "max_depth": 12,
    "learning_rate": 0.1,
}

model_overfit, auc_overfit, _ = train_and_log_xgb(
    "XGBoost_overfit_test",
    params_overfit,
    X_train_sel, X_val_sel, y_train, y_val,
)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [12:28:53] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


🏃 View run XGBoost_overfit_test at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/bf2ecb334e35427cbf4bff61eb77f6d4
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
XGBoost_overfit_test                      train_auc=1.0000  val_auc=0.9720  gap=+0.0280


## Tuned მოდელი
შემცირებული learning rate, მეტი ხე, რეგულარიზაცია.

In [33]:
params_tuned = {
    "n_estimators": 500,
    "max_depth": 7,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
}

model_tuned, auc_tuned, val_preds_tuned = train_and_log_xgb(
    "XGBoost_tuned",
    params_tuned,
    X_train_sel, X_val_sel, y_train, y_val,
)

🏃 View run XGBoost_tuned at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/567e12ae987f4dd89a959a35525b7abc
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
XGBoost_tuned                             train_auc=0.9862  val_auc=0.9506  gap=+0.0356


## feature selection-ის გარეშე
ვამოწმებთ, დაგვეხმარა თუ არა სელექშენი

In [34]:
model_all, auc_all, _ = train_and_log_xgb(
    "XGBoost_tuned_all_features",
    params_tuned,
    X_train, X_val, y_train, y_val,
)

🏃 View run XGBoost_tuned_all_features at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/41f345eaf9414bd0ada8da104f7f2a3c
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
XGBoost_tuned_all_features                train_auc=0.9871  val_auc=0.9497  gap=+0.0374


## შედეგების შედარება


In [35]:
results = {
    "baseline (300 trees, depth 6)": auc_baseline,
    "overfit (1000 trees, depth 12)": auc_overfit,
    "tuned (500 trees, depth 7)": auc_tuned,
    "tuned + all features": auc_all,
}

print("=== XGBoost Results ===")
for name, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:40s}  val_auc={auc:.4f}")

best_name = max(results, key=results.get)
best_auc = results[best_name]
print(f"\nBest: {best_name} (AUC={best_auc:.4f})")

=== XGBoost Results ===
  overfit (1000 trees, depth 12)            val_auc=0.9720
  tuned (500 trees, depth 7)                val_auc=0.9506
  tuned + all features                      val_auc=0.9497
  baseline (300 trees, depth 6)             val_auc=0.9411

Best: overfit (1000 trees, depth 12) (AUC=0.9720)


# მოდელის რეგისტრაცია
საუკეთესო XGBoost მოდელს ვინახავთ MLflow Model Registry-ში.

In [37]:
all_results = {
    "baseline":   (model_baseline, auc_baseline, selected_features),
    "overfit":    (model_overfit,  auc_overfit,  selected_features),
    "tuned":      (model_tuned,    auc_tuned,    selected_features),
    "all_feats":  (model_all,      auc_all,      X_train.columns.tolist()),
}

best_name = max(all_results, key=lambda k: all_results[k][1])
best_model, best_auc, best_features = all_results[best_name]

print(f"Best run: {best_name}  (val AUC = {best_auc:.4f})")
print(f"Features used: {len(best_features)}")

with mlflow.start_run(run_name="FINAL_XGBoost_best") as run:
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("source_run", best_name)
    mlflow.log_param("n_features", len(best_features))
    mlflow.log_metric("val_auc", best_auc)

    mlflow.xgboost.log_model(
        best_model,
        name="model",
        registered_model_name="ieee-fraud-best-model",
    )

    print(f"Registered as 'ieee-fraud-best-model'")

Best run: overfit  (val AUC = 0.9720)
Features used: 200


Registered model 'ieee-fraud-best-model' already exists. Creating a new version of this model...
2026/05/02 12:39:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ieee-fraud-best-model, version 2
Created version '2' of model 'ieee-fraud-best-model'.


Registered as 'ieee-fraud-best-model'
🏃 View run FINAL_XGBoost_best at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/8c6192bfbbf343ce8f34bea13b116029
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
